# Mathematical Modelling of Population Harvesting under a Strong Allee Effect: Equilibria, Stability, Extinction and PINN Analysis

## Project Overview

In this project, we study how harvesting affects a population that follows the strong Allee effect. This means that if the population falls below a certain minimum size, it may continue to decline even without further harvesting.

We use fishing as an example of harvesting, where the term *stock* refers to the population being studied, measured either by the number of individuals or by their total weight. *Catch* refers to the amount removed from the population per unit time.

We consider two harvesting methods:

- **Constant harvesting (Model A):** A fixed amount $H$ is removed per unit time, regardless of the current population size.
- **Proportional harvesting (Model B):** The amount removed is $hN$, where $N$ is the current population and $h$ is the harvesting-rate coefficient, also called fishing effort. Thus, the catch decreases as the population becomes smaller.

For example, if the population is 100 units, a constant harvest of 10 units per time unit remains unchanged even if the population decreases to 50. However, with proportional harvesting at $h=0.1$ per time unit, the catch decreases from 10 to 5 units per time unit.

We study the equilibrium points, stability, collapse thresholds, and extinction behaviour of these models. We then use Physics-Informed Neural Networks (PINNs) to solve the equations and estimate unknown parameters from simulated observations. All data used in this project are synthetic.

## Contributors

| Section | Names                | ID            |
| ------------ | -------------------- | ------------- |
| 1         | Raghav Singhal       | 2024ADPS0827G |
| 2         | Ashwath Chandramouli | 2024ADPS0853G |
| 3         | Devam Jatin Bhavsar  | 2024ADPS0831G |
| 4         | Mehta Prem Kalpesh   | 2024ADPS0851G |
| 5         | Rohan Agrawal        | 2024ADPS0004G |

**Running this notebook:** Open the notebook in Google Colab, connect to a CPU runtime, and select **Runtime → Run all**. The notebook automatically downloads the necessary Python code and synthetic data, runs the experiments, and displays the corresponding calculations, tables, and graphs. An internet connection is required for the initial setup, but no Google Drive connection or API key is needed. The source code, experimental data, and supporting documentation can be found in our [GitHub project repository](https://github.com/devam2006/mm-allee-harvesting).&#x20;


In [ ]:
# @title Load the project data and code
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path
from zipfile import ZipFile

_url = "https://raw.githubusercontent.com/devam2006/mm-allee-notebook-runtime/6ff183adae6093c26e663dfac254d2c071ea06a0/runtime/wp01-wp05-def74213164d849f.zip"
_expected = "def74213164d849f511f28c1ff8f581e87fef01ba67bb6b33c77934ff72ef64c"
_name = "wp01-wp05-def74213164d849f.zip"
_cache = Path(tempfile.gettempdir()) / "mm-allee-runtime" / _name
_local_candidates = [Path.cwd() / "runtime" / _name, Path.cwd() / _name, _cache]
for _candidate in _local_candidates:
    if _candidate.is_file():
        _payload = _candidate.read_bytes()
        print("Using downloaded/packaged project data.")
        break
else:
    print("Downloading the fixed public data/code snapshot...")
    with urllib.request.urlopen(_url, timeout=60) as _response:
        _payload = _response.read(10 * 1024 * 1024 + 1)
if len(_payload) > 10 * 1024 * 1024 or hashlib.sha256(_payload).hexdigest() != _expected:
    raise RuntimeError("Project data checksum mismatch. Use a fresh notebook/download package.")
_cache.parent.mkdir(parents=True, exist_ok=True)
_cache.write_bytes(_payload)
ROOT = Path(tempfile.mkdtemp(prefix="mm-allee-notebook-"))
with ZipFile(io.BytesIO(_payload)) as _archive:
    for _entry in _archive.infolist():
        _path = (ROOT / _entry.filename).resolve()
        if not _path.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Invalid project data path")
        _path.parent.mkdir(parents=True, exist_ok=True)
        _path.write_bytes(_archive.read(_entry))
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
os.chdir(ROOT)
_missing = [
    name
    for name in ["numpy", "scipy", "matplotlib", "IPython", "jax"]
    if importlib.util.find_spec(name) is None
]
if _missing:
    print("Installing missing scientific packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *_missing])
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    import matplotlib

    matplotlib.use("Agg")
print("Project ready. Run the remaining cells from the top.")

## Import Libraries and Set Up Project Paths




In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A: steady populations under a fixed catch

**Question.** With the same catch every year, where can the population stay steady, which equilibrium points are stable, and what is the largest catch before collapse is certain?

**Model A.** A well-mixed population $N(t)$ with growth rate $r$, carrying capacity $K$ and Allee threshold $A$ ($0<A<K$), harvested at a constant rate $H$:
$$\frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H.$$
Below $A$ growth is negative, so a small population shrinks even without fishing. We assume deterministic dynamics and constant parameters, and stop a simulation when $N$ reaches 0, since an extinct stock cannot be fished.

**Scaling.** With $x=N/K$, $\tau=rt$, $a=A/K$ and $q=H/(rK)$,
$$\frac{dx}{d\tau}=f(x)=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big),$$
so only $a$ and $q$ matter. We use $a=0.2$, for example $r=0.5$/year, $K=1000$ t and $A=200$ t as in Section 2.

### 1.1 Steady states and stability

A steady state $x_e$ solves $f(x_e)=0$. It is **stable** if $f>0$ just below it and $f<0$ just above it, so a small disturbance dies out. With the opposite signs it is **unstable**, and with the same sign on both sides it is **semistable**.

**Logistic baseline** ($f=x(1-x)-q$, no Allee effect). For $q<\tfrac14$ the steady states $x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ are unstable (lower) and stable (upper). They merge at $q=\tfrac14$, so the largest catch is $H_c=rK/4$. Without harvest, zero is unstable and any small population recovers.

In [ ]:
import matplotlib.pyplot as plt

from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    states = [f"x={e.x:.3f} ({e.stability})" for e in equilibria_logistic(q)]
    print(f"q={q}:", ", ".join(states) or "none, the population collapses")

### 1.2 Model A: the growth curve and the maximum catch

$G$ is negative on $(0,a)$ and forms a hill on $(a,1)$. Setting $G'(x)=0$ gives the top of the hill,
$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*).$$
Steady states are where the line at height $q$ meets $G$:

- **$q=0$:** $0$ (stable), $a$ (unstable, the minimum viable population) and $1$ (stable).
- **$0<q<q_c$:** $x_-$ between $a$ and $x^*$ (unstable) and $x_+$ between $x^*$ and 1 (stable). Below $x_-$ the population dies out; above it, it settles at $x_+$.
- **$q=q_c$:** one semistable point at $x^*$, the fold.
- **$q>q_c$:** none, so every population collapses.

For $q>0$, $f(0)=-q<0$, so zero is  no longer an equilibrium point. If the initial population is below the unstable equilibrium (x_-), it decreases and reaches zero in finite time. For $a=0.2$, $x^*\approx0.706$ and $q_c\approx0.525$, a maximum catch of about 263 t/year in the example.

### 1.3 Finding the steady states numerically

**Algorithm.**
1. Compute $x^*$ and $q_c$.
2. If $q>q_c$ there is no positive steady state; if $q=q_c$ there is one, at $x^*$.
3. If $0<q<q_c$, $f$ changes sign once on $[a,x^*]$ and once on $[x^*,1]$. Find each root with Brent's method, a standard root finder that narrows an interval around a sign change.
4. Label each root from the signs of $f$ on either side, as in 1.1.

In [ ]:
from mm_allee.equilibria import equilibria_a, fold_a

a = 0.2
x_star, q_c = fold_a(a)
print(f"x* = {x_star:.4f}, q_c = {q_c:.4f}\n")
for q in [0.0, 0.3, q_c, 0.6]:
    states = [f"x={e.x:.4f} ({e.stability})" for e in equilibria_a(a, q)]
    print(f"q={q:.4f}:", ", ".join(states) or "none, the population collapses")

### 1.4 Growth curves and the branch diagram

Filled circles are stable, open circles unstable and half-filled semistable. Arrows on the phase lines show whether the population grows or shrinks.

In [ ]:
from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** As the constant harvesting rate increases, the stable equilibrium (solid curve) and the unstable equilibrium (dashed curve), which acts as the collapse threshold, move closer together. They eventually meet at the critical point $(q_c, x^*)$. Beyond $q_c$, no positive equilibrium exists, and the population eventually becomes extinct.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** As the catch rises, the stable branch (solid) and the collapse threshold (dashed) move together and meet at the fold $(q_c,x^*)$, the gap between the equilibrium points decreases as the harvesting rate approaches its critical value.

### 1.5 Checking stability by simulation

At $q=0.3$, start $\delta=0.01$ above and below each steady state and solve the ODE (RK45, stopping at zero).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

Both starts near $x_+$ return to it. Just above $x_-$ the population recovers, and just below it reaches zero at $\tau\approx2.9$, as the stability labels predict.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong></div>

As the constant harvesting rate increases, the stable and unstable equilibrium points move closer together and eventually merge at the critical harvesting rate (q_c). A population starting near the stable equilibrium can still collapse if a sufficiently large decrease takes it below the unstable equilibrium. Therefore, harvesting close to (q_c) leaves the population more vulnerable to disturbances.

**Limitations.** No randomness (which matters most near the threshold), a simple cubic growth law, constant parameters, and an exact catch whenever $N>0$. The numbers are illustrative, not from a real stock.

## 2. Does a smaller catch prevent collapse?

**Model A takes a fixed amount each year. Model B removes an amount proportional to the current population.**
For example, At a population of 100 tonnes, a harvesting rate of 10% per year corresponds to a catch rate of 10 tonnes/year,
but only 2 tonnes from a 20-tonne population.

We answer three questions:
1. Which population levels can remain steady under Model B?
2. How can we compare the two harvesting rules fairly?
3. Does reducing harvesting after collapse allow recovery?

These are **synthetic examples**, not observations of a real fish stock.

### 2.1 Steady populations under Model B

Use the same scaled variables as Section 1: $x=N/K$ is population relative to carrying
capacity, $a=A/K$ is the Allee threshold, $\tau=rt$ is scaled time, and $e=h/r$ is harvesting effort.

$$
\frac{dx}{d\tau}=x\left[(1-x)(x/a-1)-e\right]
$$

A steady population makes the right-hand side zero. One solution is $x=0$.
For the other solutions, divide by $x$ and rearrange:

$$
x^2-(1+a)x+a(1+e)=0
$$

The two roots are

$$
x_{\pm}=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}
$$

The lower equilibrium \(x_-\) represents the minimum population required for recovery under the given harvesting effort. When harvesting increases, this threshold rises above the original Allee threshold \(a\), making recovery more difficult.
They meet at the critical effort

$$
e_c=\frac{(1-a)^2}{4a}
$$

- **Below $e_c$:** the lower root $x_-$ is the recovery threshold; the upper root $x_+$ is stable.
  Starting below $x_-$ leads towards zero; starting above it leads towards $x_+$.
- **At $e_c$:** the roots meet. The steady state attracts from above but not from below
  (called *semistable*).
- **Above $e_c$:** no positive steady population remains.

Why? Factoring gives $f_B=-x(x-x_-)(x-x_+)/a$, whose signs are negative, positive,
negative across the three positive-population intervals. Also $f'_B(0)=-(1+e)<0$,
so zero stays stable even when harvesting stops.

**Example:** with $a=0.2$ and $e=0.4$, the recovery threshold is about $0.317K$
and the stable population is about $0.883K$. Here $e_c=0.8$.
The code below runs the shared experiment and shows these branches.

In [ ]:
import csv
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

from mm_allee.wp02_experiment import run_experiment

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "configs/wp02.json").exists() and (path / "src/mm_allee").exists()
)
# Run this experiment once.
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
display(Image(filename=str(output / "wp02_b_branches.png")))

### 2.2 Compare harvesting fairly

A fixed catch $H$ (tonnes/year) and an effort $h$ (per year) have different units.
We therefore use **two comparisons**:

1. **Same starting catch:** choose $H=hN_0$. Both rules remove the same amount initially;
   Model B's catch then changes with population.
2. **Same fraction of the limit:** use half of each model's own critical harvesting level.
   This compares equal distance from the collapse limit, but the initial catches differ.

For this example, $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes, and the starting
population is 900 tonnes. We follow it for 60 years. The logistic model from Task 1
is included as a baseline.

**Mean catch = total catch / 60 years.** Total catch is the area under the catch-rate
curve: $\int_0^{60}hN(t)\,dt$ for B. For a fixed catch, harvesting stops at extinction.
The mean includes the years after shutdown.

The next table shows the important comparison results; the full CSV remains available.

In [ ]:
with (output / "wp02_comparison.csv").open() as f:
    rows = list(csv.DictReader(f))
labels = {"initial_catch": "Same starting catch", "threshold_fraction": "Half of each limit"}
outcomes = {"persistence": "Persists", "finite_extinction": "Collapses"}
lines = [
    "| Comparison | Model | Starting catch (t/year) | Mean catch (t/year) | Outcome |",
    "| --- | --- | --- | --- | --- |",
]
for row in rows:
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5:
        lines.append(
            f"| {labels[row['protocol']]} | {row['model']} | "
            f"{float(row['initial_catch']):.2f} | {float(row['mean_catch']):.2f} | "
            f"{outcomes.get(row['theoretical_outcome'], row['theoretical_outcome'])} |"
        )
display(Markdown("\n".join(lines)))

**What this shows:** at the same starting catch (131.28 tonnes/year), A and B persist,
but the logistic population collapses because its maximum fixed catch is only
125 tonnes/year. At half of each model's limit, all three persist.

This does **not** prove that Allee growth is safer: the models have different growth
curves, and the conclusion depends on how the comparison is set up.

### 2.3 Does the population recover when harvesting falls?

**Method:** start at $N=K$, increase harvesting in steps to 1.3 times its critical
level, then decrease it to zero. Carry the population forward at every step—never
restart it at the turning point. Repeat with two dwell times (1 and 3 scaled time
units per ordinary step).

After harvesting returns to zero, compare:
- **No intervention:** let the population evolve on its own.
- **Restocking:** add $0.6K=600$ tonnes, shown separately by the dashed lines.

For A, reaching zero triggers harvest shutdown. For B, a positive population
approaches zero without reaching it in finite time; $x\leq10^{-6}$ is only a
smallness marker. The horizontal axis below is $\tau=rt$; here one unit equals two years.

In [ ]:
display(Image(filename=str(output / "wp02_history.png")))

### 2.4 Main conclusion

**Reducing harvesting does not guarantee recovery.** In both tested schedules,
the unassisted populations remain near zero. Once below the Allee threshold,
natural growth is negative even without harvesting. Adding 600 tonnes crosses
the threshold and allows recovery towards $K$.

This is **history dependence with a recovery barrier**, not a closed loop that
recovers by itself. The slower schedule also means longer exposure to harvesting,
so the experiment does not isolate the effect of speed alone.

**Checks and limits:** the experiment repeats the histories with tighter solver
settings; population differences are below $2\times10^{-9}$ in the recorded run.
Very small B populations have no claimed relative accuracy. Outcomes near an
uncertain starting-population boundary are labelled unresolved. The code, full
comparison table and run settings are saved by the experiment; no real-stock
prediction is claimed.

**Remember:** a smaller catch at low population helps reduce removal, but it does
not remove the Allee recovery threshold.


## 3. Critical Harvesting Rates and Extinction Analysis

In Sections 1 and 2, we studied the equilibrium points and stability of Models A and B. We now calculate the maximum harvesting rates that allow a positive equilibrium, determine which initial populations survive, and find the time taken for extinction when the population collapses.

We use the same population models and assumptions as before. All numerical results are obtained using synthetic data, with constant parameters and no random environmental effects.

Using the dimensionless variables introduced in Section 1,

$$
x=\frac{N}{K},\quad \tau=rt,\quad
a=\frac{A}{K},\quad q=\frac{H}{rK},\quad e=\frac{h}{r},
$$

the two models are

$$
\begin{aligned}
\text{Model A:}\quad &x'=G(x)-q,\\
\text{Model B:}\quad &x'=G(x)-ex,
\end{aligned}
$$

where

$$
G(x)=x(1-x)\left(\frac{x}{a}-1\right)
$$

and $x'=dx/d\tau$.

### 3.1 Critical Harvesting Rates

**Model A: Constant Harvesting**

At equilibrium, the natural population growth must balance the amount harvested. Hence, the maximum constant harvesting rate is obtained by finding the maximum value of $G(x)$.

Differentiating,

$$
G'(x)=\frac{-3x^2+2(1+a)x-a}{a}.
$$

Setting $G'(x)=0$, the larger stationary point is

$$
x_*=\frac{1+a+\sqrt{1-a+a^2}}{3}.
$$

This point lies between $a$ and $1$, and $G''(x_*)<0$, so it gives the maximum positive natural growth rate.

Therefore, the critical harvesting rate is

$$
q_c=G(x_*), \qquad H_c=rKq_c,
$$

where $H_c$ is the critical harvesting rate in the original units.

**Model B: Proportional Harvesting**

For Model B, the amount harvested depends on the population size. At a positive equilibrium,

$$
G(x)=ex.
$$

Dividing by $x$, we obtain

$$
e=\frac{G(x)}{x}=(1-x)\left(\frac{x}{a}-1\right)=P(x).
$$

Thus, unlike Model A, we find the maximum natural growth rate per unit population.

Differentiating,

$$
P'(x)=\frac{1+a-2x}{a}.
$$

Setting $P'(x)=0$ gives $x=(1+a)/2$. Since $P''(x)=-2/a<0$, this is a maximum. Substituting it into $P(x)$ gives

$$
e_c=\frac{(1-a)^2}{4a}.
$$

Converting back to the original parameters,

$$
h_c=\frac{r(K-A)^2}{4AK}.
$$

For harvesting rates above their respective critical values, neither model has a positive equilibrium. Below these values, the population may survive, depending on its initial size.

For comparison, the standard logistic model with constant harvesting has the critical rate $H_{c,\mathrm{logistic}}=rK/4$, as derived in Section 1.

Note that $H$ and $h$ have different units. We therefore cannot compare their numerical values directly.

The following code calculates the critical harvesting rates for the parameters used in our numerical examples.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))


### 3.2 Effect of Initial Population on Survival

Even when harvesting is below the critical rate, the population may still become extinct if its initial size is too small.

To study this, we use the idea of a **basin of attraction**. A basin of attraction is the set of initial populations that approach the same equilibrium as time increases.

For Model A, when $0<q<q_c$, there are two positive equilibrium points, $x_-$ and $x_+$, where $x_-<x_+$.

By checking the sign of $x'=G(x)-q$, we find that:

- For $0<x<x_-$, the population decreases towards extinction.
- For $x_-<x<x_+$, the population increases towards $x_+$.
- For $x>x_+$, the population decreases towards $x_+$.

Thus, $x_-$ is an unstable equilibrium that separates extinction from survival, while $x_+$ is a stable equilibrium.

For Model B, the two positive equilibria are

$$
x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}.
$$

The lower equilibrium is unstable and the upper equilibrium is stable. In addition, $x=0$ is always a stable equilibrium for Model B, since

$$
f'_B(0)=-(1+e)<0.
$$

The behaviour of both models at different harvesting levels is summarised below.

| Harvesting level | Model A | Model B |
|---|---|---|
| Zero harvesting | Below $a$, population approaches zero; above $a$, it approaches $1$ | Same behaviour as A |
| Below critical rate | Below $x_-$, population reaches zero; above $x_-$, it approaches $x_+$ | Below $x_-$, population approaches zero; above $x_-$, it approaches $x_+$ |
| At critical rate | Below the merged equilibrium, population reaches zero; above it, population approaches the merged equilibrium | Below the merged equilibrium, population approaches zero; above it, population approaches the merged equilibrium |
| Above critical rate | Every positive initial population eventually reaches zero | Every positive initial population approaches zero |

At the critical harvesting rate, the two positive equilibria merge into one. This equilibrium is **semistable**, meaning that the population approaches it from above but moves away from it when starting below.

An initial population placed exactly at an equilibrium remains there.

There is also an important difference between the models at zero population. For Model A with positive constant harvesting, $x=0$ is not an equilibrium because harvesting continues at a fixed rate. We therefore stop the numerical solution when the population first reaches zero.

For Model B, the amount harvested decreases with population, so zero remains an equilibrium.

The numerical calculations also account for rounding errors near unstable and semistable equilibria. Initial values extremely close to these boundaries are marked as unresolved rather than being classified incorrectly.



### 3.3 Calculating the Time to Extinction

For Model A, we can calculate the time taken for the population to reach zero using separation of variables.

We have

$$
\frac{dx}{d\tau}=G(x)-q.
$$

Separating variables,

$$
d\tau=\frac{dx}{G(x)-q}.
$$

Integrating from the initial population $x_0$ to zero and reversing the limits,

$$
\tau_{\mathrm{ext}}
=\int_{x_0}^{0}\frac{dx}{G(x)-q}
=\int_0^{x_0}\frac{dx}{q-G(x)}.
$$

This gives the extinction time in dimensionless units.

To convert it back to the original variables, recall that

$$
x=\frac{N}{K},\qquad \tau=rt,\qquad
q=\frac{H}{rK},\qquad G(x)=\frac{g(N)}{rK}.
$$

Therefore, $dx=dN/K$ and

$$
q-G(x)=\frac{H-g(N)}{rK}.
$$

Substituting these expressions into the integral,

$$
\tau_{\mathrm{ext}}
=\int_0^{N_0}
\frac{dN/K}{(H-g(N))/(rK)}
=r\int_0^{N_0}\frac{dN}{H-g(N)}.
$$

Since $\tau=rt$, the extinction time in the original time units is

$$
\boxed{
t_{\mathrm{ext}}
=\frac{\tau_{\mathrm{ext}}}{r}
=\int_0^{N_0}\frac{dN}{H-g(N)}
}
$$

Both expressions calculate the same extinction time. The first uses dimensionless variables, while the second uses the original population and time units.

This formula is valid only when the population actually reaches zero. We must first check the initial population and harvesting rate using the stability analysis from Section 3.2.

For a declining trajectory that reaches zero, the denominator remains positive throughout the integration interval. Hence, the integral gives a finite extinction time.

When $q>q_c$, we know that $G(x)\leq q_c$. Therefore,

$$
q-G(x)\geq q-q_c>0,
$$

which gives

$$
\tau_{\mathrm{ext}}\leq\frac{x_0}{q-q_c}.
$$

This confirms that extinction occurs in finite time above the critical constant harvesting rate.

For Model B, harvesting decreases as the population becomes smaller. A positive population may approach zero, but it does not reach exactly zero in finite time. The same applies to Model A when harvesting is zero.

Thus, **reaching zero in finite time and approaching zero as time increases are two different behaviours**.

### 3.4 Numerical Solution and Accuracy Checks

We use numerical methods to calculate extinction times and verify the analytical results.

The procedure is as follows:

1. Determine whether the initial population leads to extinction using the equilibrium and stability analysis.
2. For cases where extinction occurs, calculate the extinction time by numerical integration of the formula derived above.
3. Solve the differential equation using the adaptive Runge-Kutta method RK45. For Model A, stop the solution when the population first reaches zero.
4. Compare the results with another numerical solver, DOP853, to check their accuracy.

RK45 automatically adjusts its step size using two Runge-Kutta estimates of different orders. Their difference gives an estimate of the local numerical error, which is used to choose the next step size.

For our reference calculations, we use relative tolerance $10^{-9}$, absolute tolerance $10^{-11}$, and a maximum dimensionless step size of $0.1$.

To check the accuracy of the extinction times, we compare six representative cases using numerical integration, RK45 at two relative tolerances ($10^{-8}$ and $10^{-10}$), and DOP853 at $10^{-12}$.

We accept the results when the difference in dimensionless extinction times satisfies

$$
|\Delta\tau|\leq
2\times10^{-7}+5\times10^{-7}\tau_{\mathrm{ext}}.
$$

This verifies agreement for the tested cases, but it is not an exact error bound for every possible initial condition.

The next code cell runs the calculations, checks the numerical results, and generates the basin and extinction-time plots. It also generates the reference data used in the later PINN sections.

The optional sensitivity and maximum-yield calculations are not included in this notebook.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json", ["--core-only"]),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json", []),
]
for label, script, configuration, extra in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
            *extra,
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))


**Interpretation of Results**

For the numerical examples, we use $r=0.7$, $K=100$, and $A=20$. The calculated critical values are

$$
N_*=70.55050,\qquad H_c=36.75966,\qquad h_c=0.56.
$$

For comparison, the critical constant harvesting rate for the standard logistic model is $17.5$.

**1. Model A: Basins of attraction (top-left)**

The horizontal axis shows the constant harvesting rate relative to its critical value, $H/H_c$, while the vertical axis shows the initial population relative to carrying capacity, $N_0/K$.

The blue region represents initial populations that approach a positive equilibrium. The orange region represents populations that reach zero in finite time.

The solid black curve is the unstable equilibrium, which separates survival from extinction. The dashed curve is the stable equilibrium. As harvesting increases, these two equilibria move closer together and merge at $H/H_c=1$.

Beyond this critical harvesting rate, the entire region is orange, indicating that every positive initial population eventually reaches zero.

**2. Model B: Basins of attraction (top-right)**

This graph shows the corresponding results for proportional harvesting, with $h/h_c$ on the horizontal axis.

The blue region again represents populations approaching a positive equilibrium. However, the yellow region represents populations that approach zero asymptotically, rather than reaching zero in finite time.

As in Model A, the unstable and stable positive equilibria merge at the critical harvesting rate. Beyond $h/h_c=1$, all positive initial populations decline towards zero.

Thus, both models have a critical harvesting limit, but their extinction behaviour differs because Model B's catch decreases with population size.

**3. Model A: Extinction time versus initial population (bottom-left)**

This graph shows how extinction time changes with the initial population for four different constant harvesting rates.

For harvesting below the critical rate, extinction time increases as the initial population approaches the unstable equilibrium from below. Above this boundary, the population survives, so no finite extinction time is plotted.

At the critical rate, the extinction time becomes very large as the initial population approaches the merged equilibrium from below.

When harvesting exceeds the critical rate, every positive initial population eventually reaches zero. However, extinction may still take a long time, particularly when harvesting is only slightly above the critical value.

**4. Model A: Extinction time versus constant catch (bottom-right)**

Here, the initial population is fixed at three values, $N_0=10$, $50$, and $90$, while the constant harvesting rate is varied.

For a fixed initial population that reaches zero, increasing the catch reduces the extinction time. However, the minimum catch needed to cause extinction depends on the initial population.

For $N_0=10$, the population is already below the Allee threshold, so it can decline even at relatively low harvesting rates.

For $N_0=50$, finite-time extinction occurs only once harvesting is sufficiently large to place the initial population below the unstable equilibrium.

For $N_0=90$, which is above the population level at the critical fold, extinction occurs only when $H>H_c$. The extinction time is particularly large just above the critical rate because the population declines slowly near the former equilibrium.

The vertical axes of both extinction-time graphs use a logarithmic scale.

**Numerical Verification**

The analysis produced 427 extinction records and 34 basin-classification cases. We independently compared extinction-time calculations using numerical quadrature, RK45, and DOP853.

All six validation cases passed, with the largest recorded difference in dimensionless extinction time approximately $4.56\times10^{-8}$.

Overall, the graphs show that survival depends on both the harvesting rate and the initial population. Even below the critical harvesting rate, a population may collapse if it starts below the unstable equilibrium. A long extinction time also does not necessarily imply stability.



### 3.5 Slow Decline Near the Critical Harvesting Rate

When the constant harvesting rate is slightly above its critical value, no positive equilibrium exists. However, the natural population growth can still be very close to the amount harvested near the population level $x_*$.

Since the difference between growth and harvesting is small, the population decreases very slowly in this region. It may remain close to the former equilibrium for a long time before eventually declining towards zero.

This behaviour is called a **bottleneck**.

The following graph shows population trajectories for three harvesting rates above the critical value. We observe that the closer the harvesting rate is to the critical value, the longer the population takes to collapse.

Thus, a population that changes very little over a short period is not necessarily stable. It may still become extinct if harvesting remains above the critical rate.

Here, we study this behaviour numerically rather than deriving an additional formula for the delay.


In [ ]:
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))


### 3.6 Synthetic Reference Data and Limitations

Along with the analytical calculations, we generated numerical solutions using RK45 for both harvesting models.

The reference dataset contains six synthetic trajectories with 606 sampled population values. These cover regular population behaviour, population decline or extinction, and cases close to the critical harvesting rates.

The parameter values, initial conditions, and numerical solver settings are recorded so that the results can be reproduced.

These numerical solutions are used as reference data in the later sections of the project. Section 4 studies the forward problem using Physics-Informed Neural Networks (PINNs), while Section 5 studies the inverse problem, where model parameters are estimated from simulated population observations.

The reference solutions also allow us to compare the PINN predictions with a conventional numerical method. The clean reference values used for evaluation are kept separate from the observations used for parameter fitting.

**Main conclusion.**

Our results show that both the harvesting rate and the initial population determine whether a population survives. Under constant harvesting, extinction can occur in finite time. Under proportional harvesting, a declining positive population approaches zero without reaching it in finite time.

We also find that extinction can be considerably delayed when the constant harvesting rate is just above its critical value. The numerical comparisons provide additional checks on the extinction-time calculations.

**Limitations.**

The models assume constant parameters and a well-mixed population. They do not account for environmental randomness, differences in age, or delays in population growth.

All experiments use synthetic data. Therefore, the results describe the behaviour predicted by these mathematical models and are not direct predictions for any particular real-world population.


In [ ]:
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))

## 4. Forward PINNs

### 4.1 Method and Evaluation

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$). The domain is $\tau\in[0,T]$ ($T=10$; for Model A, $T$ is the extinction time) and $x\in[0,1]$. The setting below is the **same one used for the inverse problem** (Section 5).

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network with a hard initial condition.** A fully connected network $N(s;\theta)$ with layer sizes $(1,32,32,1)$ (JAX, float64) and the $\tanh$ activation, with fixed time scaling $s=\tau/T$, defines

$$\hat x(\tau)=x_0+s\,N(s;\theta),$$

so $\hat x(0)=x_0$ **exactly for any weights**: no initial-condition loss is needed. Automatic differentiation with respect to physical time applies the chain rule $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ (with the product rule on $s\,N$).

**Loss.** At $n=300$ fixed uniform collocation times $\tau_i\in[0,T]$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L(\theta)=\frac1n\sum_i R(\tau_i)^2 .$$

**Algorithm.** (1) fixed collocation grid; (2) $\hat x$ and $d\hat x/d\tau$ by `jax.grad` + `vmap`; (3) $\mathcal L$; (4) Adam for 1000 iterations at learning rate $10^{-3}$ (about $2/7$ of the 3500-iteration budget) to reach a good region, then L-BFGS for 2500 iterations with the tolerance stop switched off, to converge. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol $10^{-12}$), separate from training. Metrics: RMSE, MAE, global relative $L^2$ (reported in scientific notation), max error, max error where $x_{\rm ref}<0.05$ (near-zero error), negative samples, runtime. A prediction containing NaN or infinity is *never scored* (it is labelled invalid and cannot count as accurate); a run that raises an exception is recorded as failed. A run is *inaccurate* if relative $L^2>5\times10^{-2}$ (threshold fixed beforehand).


In [ ]:
import csv

# Live demo (a few seconds on CPU): short run (100 Adam + 500 L-BFGS), Model B regular, seed 0, final design.
# A short budget is not the final result; compare with the curated table below.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
training = {
    "activation": "tanh",
    "lr": 1e-3,
    "n_collocation": 300,
    "adam_iters": 100,
    "lbfgs_iters": 500,
}
res = wp04.run_pinn(case, 0, {"training": training}, hit)
ev = evaluate_prediction(predict(res.params, res.config, tau), x_ref)
print({k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in ev.items()})
print(f"{res.wall_seconds:.1f} s, final physics loss {res.history['loss'][-1]:.3e}")

## 4.2 Curated Final Results (Loaded, Not Retrained)


In [ ]:
rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'near-0':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{fmt(r['max_abs_near_zero']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.2f}"
    )

**Figure:** Forward PINN vs reference (run the next cell).

**Figure:** Error and loss (run the next cell).

The vertical line in the bottom row marks the Adam to L-BFGS hand-over.

In [ ]:
import IPython.display as _figures

_figures.display(_figures.Image(filename=str(ROOT / "artifacts/figures/wp04_forward.png")))

In [ ]:
import IPython.display as _figures

_figures.display(_figures.Image(filename=str(ROOT / "artifacts/figures/wp04_error_loss.png")))

## 4.3 Interpretation

- **Final sweep (tanh, lr $10^{-3}$, seeds 0/1/42):** all 18 runs are accurate, relative $L^2$ from $2.9\times10^{-7}$ to $4.1\times10^{-3}$ (median $6.4\times10^{-6}$); the largest error is A near-fold (collapse after a bottleneck), the hardest case.
- Five runs predict slightly negative populations near zero (B decay seeds 1 and 42: 23 and 61 of 1001 samples; A extinction and A near-fold: 1 sample each). Near-zero error (largest absolute error where $x_{\rm ref}<0.05$): A extinction $8.9\times10^{-8}$ to $1.9\times10^{-7}$, B decay $7.3\times10^{-7}$ to $2.6\times10^{-6}$, A near-fold $4.8\times10^{-6}$ to $8.6\times10^{-3}$.
- In every run L-BFGS stopped at its 2500-iteration limit (not an abnormal exit), so the iteration budget is binding; longer runs were not tried.
- RK45 matches the reference to about $10^{-10}$ in $4$ to $10$ ms; each PINN run took $3.2$ to $4.6$ s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for the inverse problem (Section 5).
- For Model A the horizon $T$ is the extinction time from the reference solver: the forward PINN does **not** predict the extinction time itself.
- *Limitations:* one parameter set per regime, three seeds, setting selected on the same seeds (then confirmed on three others, below), one machine; per-seed results depend on platform and library versions (exact versions are in the manifests). A **small training loss is not evidence of accuracy**: see the diagnostic study in the companion source.


## 4.4 Held-Out Confirmation and Reproducibility

The frozen design (tanh, $(1,32,32,1)$, lr $10^{-3}$, 300 points) was run unchanged on seeds 11, 22, 33, which were **not** used to select it. All 18 runs are accurate (A near-fold $1.8\times10^{-4}$ to $2.3\times10^{-3}$ is the largest); both seed sets give **36 of 36 accurate runs**.

In [ ]:
conf = [
    r
    for r in csv.DictReader(open("artifacts/tables/wp04_confirmation_metrics.csv"))
    if r["method"] == "pinn"
]
for case in dict.fromkeys(r["case"] for r in conf):
    errs = [float(r["relative_l2"]) for r in conf if r["case"] == case]
    print(f"{case:13s} seeds 11/22/33 rel L2: " + ", ".join(f"{e:.1e}" for e in errs))
print("accurate:", sum(float(r["relative_l2"]) <= 5e-2 for r in conf), "of", len(conf))

# 5. Inverse PINNs, Identifiability, and Held-Out Validation

**Contributor:** Rohan Agrawal (WP05)

In this section, we estimate unknown parameters of strong-Allee harvesting models from discrete population observations.
We investigate two central questions:
1. **Structural Identifiability:** Are model parameters uniquely determined by ideal observation trajectories?
2. **Inference & Generalization:** How accurately can parameters and held-out future states be predicted under measurement noise using conventional ODE least squares versus Physics-Informed Neural Networks (PINNs)?

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Automatic Colab & Local path detection
if "google.colab" in sys.modules:
    for candidate in [
        Path.cwd(),
        Path("/content"),
        Path("/content/Rohan_RK45_Data_and_Code"),
        Path("/content/mm-allee-harvesting"),
    ]:
        if (candidate / "src/mm_allee").exists():
            if str(candidate / "src") not in sys.path:
                sys.path.insert(0, str(candidate / "src"))
            os.chdir(candidate)
            break
else:
    ROOT = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/mm_allee").exists()
        ),
        None,
    )
    if ROOT is not None and str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

from mm_allee.data import generate_noisy_observations
from mm_allee.inference import fit_ode_least_squares
from mm_allee.models import Parameters
from mm_allee.plotting import save_figure
from mm_allee.solver import solve_trajectory

print("Loaded shared modules successfully.")


wp05_demo_dir = ROOT / "results/wp05_rohan_demo"
wp05_demo_dir.mkdir(parents=True, exist_ok=True)

## 5.1 Model B Structural Nonidentifiability and Counterexample

Under Model B, harvesting is proportional to population size ($h N$):
$$\frac{dN}{dt} = r N \left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right) - h N$$

Expanding the cubic polynomial:
$$\frac{dN}{dt} = -(r + h) N + r\left(\frac{1}{A} + \frac{1}{K}\right) N^2 - \frac{r}{AK} N^3$$

Any trajectory observed at a single constant effort $h$ reveals at most **three** composite polynomial combinations:
$$c_1 = -(r + h), \quad c_2 = r\left(\frac{1}{A} + \frac{1}{K}\right) = \frac{r(A+K)}{AK}, \quad c_3 = -\frac{r}{AK}$$

Because there are 4 unknown physical parameters $(r, A, K, h)$ but only 3 observable coefficients, the system is **structurally nonidentifiable** from data at a single fixed effort.

In [ ]:
%matplotlib inline

# Demonstrate the exact two-parameter-set counterexample:
p1 = Parameters(r=1.0, A=0.2, K=1.0, h=0.5)
p2 = Parameters(r=1.35, A=0.3, K=0.9, h=0.15)

c1_1, c2_1, c3_1 = -(p1.r + p1.h), p1.r * (1.0 / p1.A + 1.0 / p1.K), -p1.r / (p1.A * p1.K)
c1_2, c2_2, c3_2 = -(p2.r + p2.h), p2.r * (1.0 / p2.A + 1.0 / p2.K), -p2.r / (p2.A * p2.K)

print(f"Parameter Set 1: c1={c1_1:.2f}, c2={c2_1:.2f}, c3={c3_1:.2f}")
print(f"Parameter Set 2: c1={c1_2:.2f}, c2={c2_2:.2f}, c3={c3_2:.2f}")
assert np.isclose(c1_1, c1_2) and np.isclose(c2_1, c2_2) and np.isclose(c3_1, c3_2)

# Solve trajectories from three distinct initial states for both parameter sets
d1, d2 = p1.dimensionless(), p2.dimensionless()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for n0, color in zip([0.4, 0.7, 1.1], colors, strict=True):
    traj1 = solve_trajectory(
        "B", a=d1["a"], harvest=d1["e"], x0=n0 / p1.K, tau_end=p1.r * 4.0, n_points=81
    )
    traj2 = solve_trajectory(
        "B", a=d2["a"], harvest=d2["e"], x0=n0 / p2.K, tau_end=p2.r * 4.0, n_points=81
    )
    t1, y1 = traj1.tau / p1.r, traj1.x * p1.K
    t2, y2 = traj2.tau / p2.r, traj2.x * p2.K
    ax.plot(t1, y1, label=f"Set 1 (N0={n0})", color=color, lw=2)
    ax.plot(t2, y2, "--", label=f"Set 2 (N0={n0})", color="black", alpha=0.7)

ax.set_title(
    "Model B Identifiability Counterexample: Identical Trajectories from Different Parameters"
)
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5.2 Observations, Noise Generation, and Frozen 60/20/20 Splits

We construct synthetic observations as follows:
$$N_{\text{obs}}(t_i) = N_{\text{clean}}(t_i) + \sigma \cdot N_{\text{ref}} \cdot \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$
with fixed external scale $N_{\text{ref}} = 100.0$. Observations are never silently clipped. Each trajectory has **101 observations** on $t\in[0,8]$.

Splits are strictly partitioned into contiguous time windows:
- **Train (60%, 61 points, $t\le4.8$):** used exclusively for fitting and data loss.
- **Validation (20%, 20 points, $4.88\le t\le6.4$):** never in the gradient. The clean demo and frozen benchmark disable checkpoint selection; the noisy live demo explicitly enables validation-based stopping/checkpoint selection.
- **Held-out Test (20%, 20 points, $6.48\le t\le8$):** completely frozen until final post-training evaluation.

In [ ]:
p_true = Parameters(r=0.7, K=100.0, A=20.0, H=10.0)
d_true = p_true.dimensionless()
ref_traj = solve_trajectory(
    "A", a=d_true["a"], harvest=d_true["q"], x0=45.0 / 100.0, tau_end=0.7 * 8.0, n_points=101
)
times = ref_traj.tau / 0.7
clean_pops = ref_traj.x * 100.0

# Generate clean data (sigma = 0.0)
obs_clean = generate_noisy_observations(
    times,
    clean_pops,
    trajectory_id="a_regular_clean",
    model="A",
    noise_sigma=0.0,
    noise_seed=11,
    n_ref=100.0,
)

# Generate 1% noisy data (sigma = 0.01)
obs_1pct = generate_noisy_observations(
    times,
    clean_pops,
    trajectory_id="a_regular_noisy",
    model="A",
    noise_sigma=0.01,
    noise_seed=11,
    n_ref=100.0,
)

idx_tr, idx_va, idx_te = (
    obs_1pct["train_indices"],
    obs_1pct["val_indices"],
    obs_1pct["test_indices"],
)
print(f"Train points: {len(idx_tr)} (0 to {times[idx_tr[-1]]:.2f})")
print(f"Val points:   {len(idx_va)} ({times[idx_va[0]]:.2f} to {times[idx_va[-1]]:.2f})")
print(f"Test points:  {len(idx_te)} ({times[idx_te[0]]:.2f} to {times[idx_te[-1]]:.2f})")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(0, times[idx_tr[-1]], color="#e6e6ff", label="Train Split (60%)")
ax.axvspan(times[idx_tr[-1]], times[idx_va[-1]], color="#e6f2e6", label="Val Split (20%)")
ax.axvspan(times[idx_va[-1]], times[-1], color="#ffe6e6", label="Held-out Test (20%)")
ax.plot(times, clean_pops, "k-", lw=2, label="Clean Reference")
ax.scatter(
    times,
    obs_1pct["observed_population"],
    color="darkblue",
    s=20,
    label="Noisy Observations (1% noise)",
)
ax.set_title("Model A: Observation Splits and Noise Structure")
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, ROOT / "artifacts/figures/wp05_demo_observations_1pct")
plt.show()

## 5.3 Baseline Inverse PINN on Clean Data (Ideal Parameter Recovery)

We first evaluate the Inverse PINN under ideal, noise-free conditions ($\sigma = 0.0$) to measure how closely the two-phase optimizer (Adam $\to$ L-BFGS) recovers the declared true values ($r=0.70, A=20.0, H=10.0$) from starting guesses ($r=1.0, A=30.0, H=5.0$). Numerical errors are reported rather than assuming exact recovery.

Physical parameters are constrained to valid ecological bounds via smooth bijective transformations:
$$r = \mathrm{softplus}(\tilde{r}) > 0, \quad A = K \cdot \sigma(\tilde{A}) \in (0, K), \quad H = \mathrm{softplus}(\tilde{H}) \ge 0$$
Training loss combines data fidelity on training observations ($t \le 4.8$) and the physics ODE residual evaluated at 300 collocation points. The network uses 2 hidden layers of 32 units with $\tanh$ activation and exact hard initial conditions $x(0) = x_0$.

**Live illustrations versus frozen benchmark.** These two single-case demos use parameter learning rate $5\times10^{-3}$, physics weight $0.5$ and at most 3000 L-BFGS iterations, with 500 Adam steps for clean data and up to 1200 for noisy data. Each fit has a 120-second CPU budget. The recorded benchmark in Section 5.5 retains its original shared configuration (parameter learning rate $10^{-3}$, physics weight $0.1$, 1000 Adam and 2500 L-BFGS steps, validation selection off); it is loaded, not retrained here.

In [ ]:
try:
    import jax  # noqa: F401  (availability check)
    import jax.numpy as jnp
    import matplotlib.pyplot as plt
    from IPython.display import display
    from jax.flatten_util import ravel_pytree

    import mm_allee.pinn.inverse as inv_mod
    from mm_allee.pinn.inverse import InversePINNTrainer, TrainableParameters
    from mm_allee.pinn.network import NetworkConfig, init_params

    known = {"known_initial_population": 45.0, "known_K": 100.0}
    ncfg = NetworkConfig(layers=(1, 32, 32, 1), activation="tanh", seed=1011)
    tp = TrainableParameters(
        "A",
        fit_params=["r", "A", "H"],
        known_conditions={"known_K": 100.0},
        initial_guesses={"r": 1.0, "A": 30.0, "H": 5.0},
    )

    # Track parameter trajectory across both Adam exploration and L-BFGS refinement
    dummy_net = init_params(ncfg)
    _, unravel_fn = ravel_pytree((dummy_net, tp.raw_params))
    lbfgs_history = {p: [] for p in tp.fit_params}

    orig_minimize = inv_mod.minimize

    def tracking_minimize(fun, x0, *args, **kwargs):
        user_cb = kwargs.get("callback")

        def wrapped_cb(intermediate_result):
            x_vec = getattr(intermediate_result, "x", intermediate_result)
            cand = unravel_fn(jnp.asarray(x_vec))
            phys = tp.get_physical_parameters(cand[1])
            for p in tp.fit_params:
                lbfgs_history[p].append(float(phys[p]))
            if user_cb is not None:
                # Preserve native logging and cancellation; do not swallow errors.
                user_cb(intermediate_result)

        kwargs["callback"] = wrapped_cb
        return orig_minimize(fun, x0, *args, **kwargs)

    trainer = InversePINNTrainer(
        network_config=ncfg,
        trainable_params=tp,
        lr_net=1e-3,
        lr_params=5e-3,
        lambda_data=1.0,
        lambda_phys=0.5,
        lbfgs_iters=3000,
        lbfgs_ftol=0.0,
        lbfgs_gtol=0.0,
        validation_selection=False,
        time_budget_s=120.0,
    )

    try:
        inv_mod.minimize = tracking_minimize
        pinn_clean = trainer.fit(
            obs_clean,
            known_conditions=known,
            epochs=500,
            patience=500,
            n_collocation=300,
            seed=1011,
        )
    finally:
        inv_mod.minimize = orig_minimize

    assert inv_mod.minimize is orig_minimize
    assert len(lbfgs_history["r"]) == pinn_clean.training_history["phase"].count("lbfgs")
    print(f"PINN Clean Fit Completed: {pinn_clean.status} in {pinn_clean.wall_seconds:.1f}s")
    print(
        "L-BFGS exit:", pinn_clean.optimizer["lbfgs_outcome"], pinn_clean.optimizer["lbfgs_message"]
    )
    print("Parameter Recovery on Clean Data:")
    for p in ["r", "A", "H"]:
        true_v = getattr(p_true, p)
        est_v = pinn_clean.estimated_parameters[p]
        rel_err = abs(est_v - true_v) / true_v * 100
        print(f"  {p}: True = {true_v:.2f}, PINN Est = {est_v:.4f} (Error: {rel_err:.2f}%)")

    # Combine Adam and L-BFGS history so curves end exactly at final values
    adam_len = len(pinn_clean.training_history["param_r"])
    full_history = {}
    for p in tp.fit_params:
        adam_vals = list(pinn_clean.training_history[f"param_{p}"])
        lbfgs_vals = list(lbfgs_history[p])
        combined = adam_vals + lbfgs_vals
        if not combined:
            raise RuntimeError("No parameter trajectory was recorded")
        full_history[p] = combined

    total_steps = list(range(1, len(full_history["r"]) + 1))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.2))

    # Parameter r
    ax1.plot(total_steps, full_history["r"], color="#1f77b4", lw=2, label="Estimated r")
    ax1.axhline(p_true.r, color="black", linestyle="--", lw=1.5, label=f"True r = {p_true.r:.2f}")
    if adam_len < len(total_steps):
        ax1.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax1.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["r"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['r']:.4f}",
    )
    ax1.set_title("Growth Rate (r) Convergence")
    ax1.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax1.set_ylabel("Parameter Value")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Parameter A
    ax2.plot(total_steps, full_history["A"], color="#2ca02c", lw=2, label="Estimated A")
    ax2.axhline(p_true.A, color="black", linestyle="--", lw=1.5, label=f"True A = {p_true.A:.2f}")
    if adam_len < len(total_steps):
        ax2.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax2.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["A"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['A']:.4f}",
    )
    ax2.set_title("Allee Threshold (A) Convergence")
    ax2.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax2.set_ylabel("Parameter Value")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Parameter H
    ax3.plot(total_steps, full_history["H"], color="#ff7f0e", lw=2, label="Estimated H")
    ax3.axhline(p_true.H, color="black", linestyle="--", lw=1.5, label=f"True H = {p_true.H:.2f}")
    if adam_len < len(total_steps):
        ax3.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax3.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["H"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['H']:.4f}",
    )
    ax3.set_title("Harvest Quota (H) Convergence")
    ax3.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax3.set_ylabel("Parameter Value")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.suptitle("Clean Data: Recorded Parameter Trajectory (Model A)", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, ROOT / "artifacts/figures/wp05_demo_clean_parameter_trajectory")
    with (wp05_demo_dir / "clean_parameter_trajectory.csv").open("w", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(["step", "phase", "r", "A", "H"])
        for i, step in enumerate(total_steps):
            writer.writerow(
                [
                    step,
                    "adam" if i < adam_len else "lbfgs",
                    *[full_history[p][i] for p in ["r", "A", "H"]],
                ]
            )
    display(fig)
    plt.close(fig)
except ImportError:
    print(
        "JAX is not installed in this environment; skipping PINN training. Full code is implemented in mm_allee.pinn."
    )

## 5.4 Parameter Estimation Under Measurement Noise: ODE Least Squares vs PINN

We now evaluate both methods on **1% measurement noise** ($\sigma = 0.01$, noise standard deviation $= 1.0$) starting from the identical initial guesses ($r=1.0, A=30.0, H=5.0$):
- **Conventional ODE Least Squares (`ODE_LS`):** Integrates the ODE with adaptive RK45 and optimizes parameters using SciPy's Trust Region Reflective algorithm on the training split ($t \le 4.8$).
- **Inverse PINN:** Jointly optimizes network weights and physical parameters using the two-phase Adam + L-BFGS optimizer on the training split and collocation grid.

Both methods run on the exact same noisy observations to provide a direct, fair comparison.

In [ ]:
# 1. Conventional ODE Least-Squares Fit
ode_res = fit_ode_least_squares(
    "A",
    observations=obs_1pct,
    known_conditions={"known_K": 100.0, "known_initial_population": 45.0},
    fit_params=["r", "A", "H"],
    initial_guess={"r": 1.0, "A": 30.0, "H": 5.0},
    n_ref=100.0,
)

# 2. Inverse PINN Fit
pinn_noisy_res = None
try:
    import jax  # noqa: F401

    from mm_allee.pinn.inverse import InversePINNTrainer, TrainableParameters
    from mm_allee.pinn.network import NetworkConfig

    ncfg = NetworkConfig(layers=(1, 32, 32, 1), activation="tanh", seed=1011)
    tp = TrainableParameters(
        "A",
        fit_params=["r", "A", "H"],
        known_conditions={"known_K": 100.0},
        initial_guesses={"r": 1.0, "A": 30.0, "H": 5.0},
    )
    trainer_noisy = InversePINNTrainer(
        network_config=ncfg,
        trainable_params=tp,
        lr_net=1e-3,
        lr_params=5e-3,
        lambda_data=1.0,
        lambda_phys=0.5,
        lbfgs_iters=3000,
        lbfgs_ftol=0.0,
        lbfgs_gtol=0.0,
        validation_selection=True,  # Rohan's noisy-demo checkpoint policy; benchmark remains False.
        time_budget_s=120.0,
    )
    pinn_noisy_res = trainer_noisy.fit(
        obs_1pct,
        known_conditions={"known_initial_population": 45.0, "known_K": 100.0},
        epochs=1200,
        n_collocation=300,
        seed=1011,
    )
except ImportError:
    pass

# 3. Comparative Summary Table
print("=" * 86)
print(f"{'Model A Parameter Recovery Under 1% Measurement Noise (sigma = 0.01)':^86s}")
print("=" * 86)
print(
    f"{'Parameter':15s}{'True Value':>12s}{'ODE_LS Estimate (Error)':>30s}{'PINN Estimate (Error)':>28s}"
)
print("-" * 86)

for p in ["r", "A", "H"]:
    tv = getattr(p_true, p)
    ode_v = ode_res.estimated_parameters[p]
    ode_err = abs(ode_v - tv) / tv * 100
    ode_str = f"{ode_v:.4f} ({ode_err:.2f}%)"
    if pinn_noisy_res is not None:
        pinn_v = pinn_noisy_res.estimated_parameters[p]
        pinn_err = abs(pinn_v - tv) / tv * 100
        pinn_str = f"{pinn_v:.4f} ({pinn_err:.2f}%)"
    else:
        pinn_str = "N/A (JAX required)"
    print(f"{p:15s}{tv:>12.2f}{ode_str:>30s}{pinn_str:>28s}")

print("-" * 86)
ode_time_str = f"{ode_res.wall_seconds:.3f}s"
pinn_time_str = f"{pinn_noisy_res.wall_seconds:.2f}s" if pinn_noisy_res is not None else "N/A"
print(f"{'Runtime':15s}{'-':>12s}{ode_time_str:>30s}{pinn_time_str:>28s}")

ode_rmse_str = f"{ode_res.test_metrics['rmse']:.4f}"
pinn_rmse_str = (
    f"{pinn_noisy_res.test_metrics['rmse']:.4f}" if pinn_noisy_res is not None else "N/A"
)
print(f"{'Test RMSE':15s}{'-':>12s}{ode_rmse_str:>30s}{pinn_rmse_str:>28s}")
print("=" * 86)

print("Noisy-demo validation selection:", trainer_noisy.validation_selection)
print(
    "L-BFGS exit:",
    pinn_noisy_res.optimizer["lbfgs_outcome"],
    pinn_noisy_res.optimizer["lbfgs_message"],
)
demo_record = {
    "scope": "Two illustrative Model A regular fits; not a rerun of the frozen benchmark",
    "source": "Rohan's supplied WP05 notebook, integrated without WP01-WP04 changes",
    "known_conditions": {"known_K": 100.0, "known_initial_population": 45.0},
    "true_parameters": {p: getattr(p_true, p) for p in ["r", "A", "H"]},
    "initial_guesses": {"r": 1.0, "A": 30.0, "H": 5.0},
    "noise_seed": 11,
    "fit_seed": 1011,
    "n_ref": 100.0,
    "demo_settings": {
        "lr_net": 1e-3,
        "lr_params": 5e-3,
        "lambda_phys": 0.5,
        "n_collocation": 300,
        "lbfgs_iters": 3000,
        "time_budget_s": 120.0,
    },
    "clean": {
        "noise_sigma": 0.0,
        "adam_epochs_requested": 500,
        "validation_selection": False,
        "estimates": pinn_clean.estimated_parameters,
        "optimizer": pinn_clean.optimizer,
        "status": pinn_clean.status,
        "success": pinn_clean.success,
        "wall_seconds": pinn_clean.wall_seconds,
    },
    "noisy": {
        "noise_sigma": 0.01,
        "adam_epochs_requested": 1200,
        "validation_selection": True,
        "estimates": pinn_noisy_res.estimated_parameters,
        "optimizer": pinn_noisy_res.optimizer,
        "status": pinn_noisy_res.status,
        "success": pinn_noisy_res.success,
        "test_rmse": pinn_noisy_res.test_metrics["rmse"],
        "wall_seconds": pinn_noisy_res.wall_seconds,
    },
    "ode_noisy": {
        "estimates": ode_res.estimated_parameters,
        "status": ode_res.status,
        "test_rmse": ode_res.test_metrics["rmse"],
        "wall_seconds": ode_res.wall_seconds,
    },
    "trajectory_rows": len(total_steps),
    "adam_trace_rows": adam_len,
    "lbfgs_trace_rows": len(lbfgs_history["r"]),
}
(wp05_demo_dir / "summary.json").write_text(json.dumps(demo_record, indent=2) + "\n")
print("WP05 demo artifacts:", wp05_demo_dir)

## 5.5 Final Sweep Benchmark (108 Recorded Runs)

The full benchmark experiment evaluates **108 total fits** (54 PINN + 54 ODE least squares) spanning:
1. **6 Dynamical Regimes:**
   - **Model A:** `a_regular` (strong-Allee persistence), `a_near_fold` (near saddle-node tipping point), `a_extinction` (collapse below Allee threshold).
   - **Model B:** `b_regular` (proportional harvest persistence), `b_near_fold` (near threshold), `b_decline` (steep transient decline).
2. **3 Noise Levels:** dimensionless $\sigma \in \{0.0, 0.01, 0.05\}$ with noise standard deviation $\sigma N_{\text{ref}}$ (clean, 1%, and 5% additive Gaussian noise).
3. **3 Random Seeds:** Seeds `{11, 22, 33}` per condition to verify stochastic reproducibility.

The table below summarizes the **clean-data baseline ($\sigma = 0$)** across the 3 seeds for each regime:
- **Parameter Recovered (PINN):** Counts how many of the 3 random seeds successfully recovered **every** unknown parameter within 5% relative error ($\le 0.05$).
- **PINN / ODE_LS Median Error:** The median across all 3 seeds of the worst-case parameter relative error for that case.

**Optimizer reporting.** The recorded benchmark has 54 PINN iteration-limit exits. Its legacy `converged` label denotes training completion, not reaching an L-BFGS tolerance. The clean/noisy live demos above use different budgets and report their own optimizer exits.

In [ ]:
import csv  # noqa: F811  (an earlier fragment imports csv in the assembled notebook)
import statistics

rows = list(csv.DictReader(open("artifacts/tables/wp05_parameter_metrics.csv")))
fits = {}
for r in rows:
    if float(r["noise_sigma"]) == 0.0:
        fits.setdefault((r["method"], r["run_id"]), {})[r["parameter"]] = float(r["relative_error"])

cases = ["a_regular", "b_regular", "a_extinction", "b_near_fold", "a_near_fold", "b_decline"]
print(
    f"{'Dynamical Case':16s}{'Parameter Recovered (PINN)':>28s}{'PINN Median Err':>18s}{'ODE_LS Median Err':>20s}"
)
print("-" * 84)

total = 0
for c in cases:
    pinn = [max(v.values()) for (m, rid), v in fits.items() if m == "PINN" and f"wp05_{c}_" in rid]
    ode = [max(v.values()) for (m, rid), v in fits.items() if m == "ODE_LS" and f"wp05_{c}_" in rid]
    ok = sum(e <= 0.05 for e in pinn)
    total += ok
    print(
        f"{c:16s}{f'{ok}/3 seeds':>28s}{statistics.median(pinn):>18.1e}{statistics.median(ode):>20.1e}"
    )

print("-" * 84)
print(f"Total PINN cases fully recovered (all parameters <= 5% error): {total}/18 clean fits")

**Figure:** Parameter recovery versus noise (run the next cell).

**Figure:** Held-out windows, Model A regular (run the next cell).

The first figure averages over all six cases and seeds, so its error bars are dominated by the badly conditioned cases. The second shows Model A regular (top row noise-free, bottom row noisy); the regular trajectories are at equilibrium by $t=4.8$, so the held-out window is an undemanding physics-supported prediction test: collocation covers the full $[0,8]$ interval, including the held-out window, so this is not extrapolation outside the physics domain.

In [ ]:
import IPython.display as _figures

_figures.display(_figures.Image(filename=str(ROOT / "artifacts/figures/wp05_recovery_noise.png")))

In [ ]:
import IPython.display as _figures

_figures.display(_figures.Image(filename=str(ROOT / "artifacts/figures/wp05_heldout_failure.png")))

### 5.6 Diagnostic Analysis: Why 8 PINN Fits Failed and Why ODE_LS Outperforms PINNs

Across the 18 clean-data fits ($\sigma = 0$), **`ODE_LS` recovers 18/18 within the declared 5% criterion**, whereas **PINN recovers 10/18** ($a\_regular$: 3/3, $b\_regular$: 3/3, $b\_near\_fold$: 3/3, $a\_extinction$: 1/3, $a\_near\_fold$: 0/3, $b\_decline$: 0/3).

This performance divergence highlights measured differences between classical numerical ODE solvers and neural physics-informed optimization:

---

#### 1. Why Eight PINN Fits Did Not Meet the Parameter-Recovery Criterion While the Other 10 Succeeded?

The 18 benchmark runs give the following recovery counts; conditioning alone does not decide success:

* **The 10 Successful Fits ($a\_regular$ 3/3, $b\_regular$ 3/3, $b\_near\_fold$ 3/3, $a\_extinction$ 1/3):**
  * **$a\_regular$ and $b\_regular$ (6/6 recovered):** Trajectories exhibit smooth, monotonic sigmoidal growth with low Jacobian condition numbers ($\kappa(J) \in [62, 340]$). The physical parameters $(r, A, H/h)$ exert distinct, decoupled influences across the transient phase, allowing data and physics loss gradients to remain balanced.
  * **$b\_near\_fold$ (3/3 recovered despite large condition number $\kappa \approx 10^3$):** A large condition number indicates that the loss surface is *ill-conditioned* (an elongated, narrow valley), **not mathematically unidentifiable**. In Model B, proportional harvesting ($hN$) scales with population and factors out $N$. The trajectory retains dynamic curvature over time. Limited-memory quasi-Newton refinement can help navigate poorly conditioned directions, but this study did not measure an exact Hessian or isolate the cause of that success.

* **The 8 Unrecovered Fits ($a\_near\_fold$ 3 failures, $b\_decline$ 3 failures, $a\_extinction$ 2 failures):**
  * **$a\_near\_fold$ (0/3 recovered):** Fixed quota harvesting near the saddle-node fold creates a severe dynamic bottleneck where $dN/dt \approx 0$ for an extended duration ("ghost of the fold"). Trajectory sensitivity collapses: $r$ and $A$ become strongly collinear ($r/A \approx \text{const}$). The recorded fits have small trajectory errors and large parameter errors. Flat dynamics and correlated sensitivities are plausible explanations; gradient magnitudes were not measured in this experiment.
  * **$b\_decline$ (0/3 recovered) and $a\_extinction$ (2/3 failed):** Both cases feature steep, rapid transient collapses followed by a long, flat tail near zero population. Limited transient information and collocation resolution are possible contributors, but their causal effect on these inverse failures has not been isolated here.

---

#### 2. Why Does `ODE_LS` Outperform PINNs Across All Cases?

| Metric / Property | Classical `ODE_LS` | Physics-Informed Neural Network (PINN) |
| :--- | :--- | :--- |
| **Search Space** | Strictly **3 dimensions** ($\theta = [r, A, H] \in \mathbb{R}^3$) | **1,000+ dimensions** ($W, b \in \mathbb{R}^{1000}$ plus $\theta \in \mathbb{R}^3$) |
| **Physics Enforcement** | **Numerical ODE constraint:** Adaptive RK45 with declared solver tolerances; solver error is not zero or machine precision. | **Soft penalty:** Approximated as a secondary loss term $\lambda_{\text{phys}} \mathcal{L}_{\text{phys}}$ evaluated at discrete collocation points. |
| **Loss Landscape** | Bounded nonlinear least squares using Trust Region Reflective. These results do not establish convexity or a unique global optimisation basin. | Multi-objective composite landscape with saddle points, flat plateaus, and spurious local minima. |
| **Gradient Dynamics** | The implementation estimates the residual Jacobian numerically; analytical sensitivities are not supplied to this optimiser. | **Gradient Stiffness / Spectral Bias:** Network weights $W$ fit observation points ($\mathcal{L}_{\text{data}} \to 0$) far faster than physical parameters $\theta$ can minimize the ODE residual ($\mathcal{L}_{\text{phys}}$), trapping $\theta$ in suboptimal values. |

> **Key Scientific Takeaway:** Parameter Error $\ne$ Trajectory Error. A PINN can fit observation points to $10^{-4}$ trajectory residual while parameter estimates remain 30% to 100% in error. `ODE_LS` avoids this failure mode because it directly evaluates the forward ODE vector field rather than an unconstrained neural surrogate.

---

#### 3. How Can the 8 Failing PINN Fits Be Recovered?

The following are candidate follow-up methods, not demonstrated fixes for these eight runs. They require new controlled experiments:
1. **Adaptive Loss Weighting (SoftAdapt / GradNorm):** Instead of static weights ($\lambda_{\text{data}}=1.0, \lambda_{\text{phys}}=0.5$), dynamically balance the gradients $\nabla_W \mathcal{L}_{\text{data}}$ and $\nabla_\theta \mathcal{L}_{\text{phys}}$ at each epoch so neither objective dominates.
2. **Residual-based Adaptive Collocation (RAR):** Dynamically concentrate collocation points in the steep collapse and inflection regions rather than distributing them uniformly across time.
3. **Curriculum Training / Time-Domain Decomposition:** Train the network sequentially over advancing time windows ($[0, t_1] \to [0, t_2] \to [0, t_{\text{end}}]$) to resolve initial transient dynamics before optimizing on flat asymptotic equilibria.

**Scope and sources.** Exact N0 is independently known in these experiments, including noisy runs. The ODE observation model uses declared zero padding after candidate extinction. Mechanistic explanations above are hypotheses where not directly tested. See [SciPy L-BFGS-B](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html) for the limited-memory approximation and [SciPy least_squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html) for the implemented TRF method.

## 5.7 Key Scientific Takeaways

1. **Parameter Error $\ne$ Trajectory Error:** A PINN can fit observations to $10^{-4}$ trajectory residual while parameter estimates remain substantially inaccurate in ill-conditioned regimes.
2. **Classical ODE Baselines are Strong:** For low-dimensional scalar ODEs, SciPy nonlinear least squares is faster, more robust, and recovers parameters across all six regimes on clean data.
3. **Structural Identifiability Precedes Optimization:** Model B with all four parameters $(r,A,K,h)$ unknown is structurally ambiguous; no optimization algorithm can break this ambiguity without independent constraints (such as known carrying capacity $K$).
4. **Noise Floor as a Convergence Metric:** Comparing training RMSE to the noise standard deviation ($\text{RMSE} / \sigma N_{\text{ref}} \approx 0.9$) measures fit relative to the noise scale; it does not by itself establish parameter accuracy or extraction of all available information.


## Reproducibility, Sources and Acknowledgements

The calculations and short demonstrations in this notebook can be executed directly in Google Colab. The larger PINN experiments were performed separately, and their recorded results are loaded into the notebook for comparison and analysis. These results are not retrained every time the notebook is run.

**Source code and data:** The complete project implementation is available in our [GitHub repository](https://github.com/devam2006/mm-allee-harvesting). It contains the Python implementations of the mathematical models, numerical methods and PINNs, along with experiment scripts, parameter configurations, generated results, figures and tests. The notebook automatically downloads the required supporting files from a fixed version of the code and data.

All population data used in this project are synthetic and generated from the mathematical models described in the notebook. The project repository also contains the supporting documentation and references.

**AI assistance:** We used ChatGPT and OpenAI Codex to assist with mathematical explanations, implementation, testing, verification, integration and preparation of the notebook and project report. Claude Code assisted with the forward PINN implementation and experiments, while Antigravity AI Assistant assisted with the inverse PINN implementation and related experiments.
